# Project-Based Learning (PBL): Pattern Recognition
## Topic: Similarity Measurement Using Distance Metrics
### Comprehensive Implementation and Comparative Benchmark of Euclidean, Manhattan, and Cosine Distance Measures

---

### **Executive Summary**
Distance and similarity measures are the foundational geometric primitives in Pattern Recognition, Computer Vision, Natural Language Processing, and Machine Learning. The choice of distance metric directly influences clustering geometry, classification decision boundaries, and retrieval fidelity.

This notebook demonstrates:
1. **Mathematical derivations & properties** of $L_2$ (Euclidean), $L_1$ (Manhattan), and Cosine similarity.
2. **From-scratch vectorized implementations** in Python/NumPy.
3. **From-scratch K-Nearest Neighbors (KNN)** classification & similarity search engines.
4. **Rigorous empirical evaluation** across low-dimensional dense, medium-dimensional numerical, high-dimensional image pixel, and sparse high-dimensional text datasets.
5. **Theoretical analysis of the *Curse of Dimensionality*** and distance concentration.

## 1. Mathematical Formulations

Given two $D$-dimensional feature vectors $\mathbf{u}, \mathbf{v} \in \mathbb{R}^D$:

### 1.1 Euclidean Distance ($L_2$ Norm)
$$d_{L_2}(\mathbf{u}, \mathbf{v}) = \|\mathbf{u} - \mathbf{v}\|_2 = \sqrt{\sum_{i=1}^{D} (u_i - v_i)^2}$$

### 1.2 Manhattan Distance ($L_1$ Norm / City Block)
$$d_{L_1}(\mathbf{u}, \mathbf{v}) = \|\mathbf{u} - \mathbf{v}\|_1 = \sum_{i=1}^{D} |u_i - v_i|$$

### 1.3 Cosine Similarity and Cosine Distance
$$\text{Cosine Similarity}(\mathbf{u}, \mathbf{v}) = \cos(\theta) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|_2 \|\mathbf{v}\|_2} = \frac{\sum_{i=1}^D u_i v_i}{\sqrt{\sum_{i=1}^D u_i^2}\sqrt{\sum_{i=1}^D v_i^2}}$$
$$d_{\text{cos}}(\mathbf{u}, \mathbf{v}) = 1.0 - \text{Cosine Similarity}(\mathbf{u}, \mathbf{v})$$

In [ ]:
import sys
import os
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.metrics import euclidean_distance, manhattan_distance, cosine_similarity, cosine_distance
from src.knn_model import CustomKNNClassifier, SimilaritySearchEngine
from src.datasets_loader import get_iris_dataset, get_wine_dataset, get_digits_dataset, get_text_dataset
from src.evaluation import benchmark_single_dataset, generate_benchmark_summary_table
from src.visualizer import plot_unit_balls_geometry, plot_decision_boundaries

print("All modules and dependencies loaded successfully!")

## 2. Interactive Distance Metric Verification
Let us test the behavior of distance metrics on collinear, orthogonal, and scaled vector pairs.

In [ ]:
# Collinear vectors (Differ only in magnitude)
u_collinear = np.array([1.0, 2.0, 3.0])
v_collinear = np.array([3.0, 6.0, 9.0])

print("--- Collinear Vectors ---")
print(f"Euclidean Distance : {euclidean_distance(u_collinear, v_collinear):.4f}")
print(f"Manhattan Distance : {manhattan_distance(u_collinear, v_collinear):.4f}")
print(f"Cosine Similarity  : {cosine_similarity(u_collinear, v_collinear):.4f}")
print(f"Cosine Distance    : {cosine_distance(u_collinear, v_collinear):.4f}")
print("Observation: Cosine distance is exactly 0.0 because the vectors share identical directional orientation!")

## 3. Geometric Iso-Distance Topologies (Unit Balls)
Visualizing the difference between $L_1$ (diamond), $L_2$ (circle), and Cosine angular rays.

In [ ]:
plot_unit_balls_geometry()
plt.show()

## 4. Multi-Dataset Experimental Benchmarking

In [ ]:
datasets = {
    'iris': get_iris_dataset(),
    'wine': get_wine_dataset(),
    'digits': get_digits_dataset(),
    'text': get_text_dataset()
}

all_results = {}
for name, dset in datasets.items():
    all_results[name] = benchmark_single_dataset(dset, k_neighbors=5)

df_summary = generate_benchmark_summary_table(all_results)
df_summary

## 5. Key Findings and Decision Guidelines

| Data Domain / Scenario | Best Metric | Rationale |
|---|---|---|
| **Low-D Continuous Numerical (Iris)** | **Euclidean ($L_2$) / Manhattan ($L_1$)** | Preserves true spatial geometry and physical scale. |
| **High-D Image Pixel Grids (Digits)** | **Euclidean ($L_2$)** | Measures direct pixel-by-pixel intensity alignment. |
| **High-D Sparse Text (TF-IDF / BoW)** | **Cosine Distance** | Invariant to document length variations and word frequency scaling. |
| **Outlier-Prone Feature Spaces** | **Manhattan ($L_1$)** | Linear error penalty avoids dominating squares of extreme outliers. |
| **Very High Dimensional Spaces ($D > 500$)** | **Manhattan / Fractional $L_p$** | Mitigates the Curse of Dimensionality distance concentration effect. |